# Проверка четырёх моделей на сохранённых данных

Порядок работы: запустите этот ноутбук на уже сохранённых CSV из `data/mock/`. Все четыре модели получают одни данные. `history_enrichment.ipynb` нужно выполнять только при отсутствии выгрузки или намеренном изменении параметров генерации; для событийной проверки повторная генерация не нужна.

Основной событийный отчёт: три известных события, продажи соответствующих курсов, вклад выбранных постов и чувствительность к сдвигу периодов. D7/D14 и календарь лидеров сохранены ниже как дополнительная диагностика.

Используется `core.attribution.compute`, без записи в БД. Покупки исторические, касания вымышлены. Категории служат для описания постов, веса ими не меняются. Затраты неизвестны, ROMI не рассчитывается. Проверяется расчёт атрибуции, а не интерфейс или вся платформа.


## Цель и гипотеза событийной проверки

Проверяем, выделяют ли модели атрибуции посты, связанные с известными событиями и всплесками продаж. Предполагаем, что анонсы и напоминания помогают заинтересованным пользователям совершить покупку, поэтому ожидаем высокого вклада этих публикаций в выручку соответствующих курсов вблизи события.

Это проверка согласованности с событийной гипотезой. Даты и содержание публикаций известны, но их причинное влияние на покупки не установлено. Продажи исторические, касания синтетические. Приписанная выручка не равна дополнительной выручке; без расходов нельзя сделать вывод о выгодности рекламы.

### Выбранные события

| Событие | Связанные посты | Продажи | Период оценки, МСК |
|---|---|---|---|
| Завершение скидок СТАРТ, 9–10 августа | №1836, 1837 — окончание акции; №1838, 1840 — продление и последние часы | Курсы линейки СТАРТ | 9–11 августа 2026 включительно |
| Анонс линейки ПРО, 22 августа | №1859 — объявление пяти направлений | Аналитика ПРО, ML ПРО, Backend ПРО, Алгоритмы ПРО, AI-агенты | 22–28 августа 2026 включительно |
| Дедлайн отбора в Т-Банк, 6 сентября | №1878 от 3 сентября; №1880 от 6 сентября — напоминания с предложением разборов на ПРО | Курсы ПРО, соответствующие направлениям отбора | 31 августа — 6 сентября 2026 включительно |

### План анализа на сохранённых данных

Шаги реализованы в разделе «Событийная проверка». Дополнительные таблицы D7/D14 решают отдельную задачу сравнения возраста постов.

1. Проверить тексты выбранных постов и соответствие продуктов вручную. Автоматическая категория не определяет принадлежность поста событию. Список событий и постов закрепить до сравнения рейтингов.
2. Загрузить существующие `orders.csv`, `order_items.csv`, `touches.csv`, `placements.csv` и `manifest.json`. Позиции заказов позволяют выделить выручку нужных курсов, не включая другие продукты из того же заказа. Генератор и сохранённые касания не менять.
3. Показать ежедневные продажи выбранных курсов, даты публикаций и события; добавить неделю до периода для контекста. При неполном наблюдении явно отметить доступные дни. Отсутствие всплеска также считать результатом.
4. Использовать одинаковые сохранённые данные и параметры атрибуции для четырёх моделей. Сначала рассчитать распределение, затем выделить нужные покупки и продукты, сохранив веса. Покупки до публикации не приписывать этому посту.
5. Сопоставить место каждого выбранного поста и полученную долю выручки периода; для серии напоминаний показать суммарный вклад группы. Сравнивать со всеми допустимыми постами, а не только с выбранными анонсами. Высокое место — ожидаемый результат гипотезы, а не условие настройки генератора.
6. Проверить чувствительность к небольшому сдвигу границ периодов на том же наборе данных. Повторения с другими генерациями оставить отдельным необязательным этапом: основной анализ не требует новых данных.

Итог этапа: три графика «продажи + события + посты» и одна таблица сравнения моделей. Календарные периоды событий оцениваются отдельно от возрастных окон D7/D14; ограничение общей когорты D14 не должно исключать сентябрьские напоминания из событийной проверки.


In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'data/raw/base.csv').exists())
sys.path.insert(0, str(ROOT/'src'))
from core.attribution import compute
MODELS = ['first', 'last', 'linear', 'position']
# Используем существующую выгрузку; повторная генерация не требуется.
DATA_DIR = ROOT/'data/mock'
input_manifest = json.loads((DATA_DIR/'manifest.json').read_text())
PARAMETERS = input_manifest['parameters']
BASE_SEED = PARAMETERS['SEED']
WINDOW_DAYS = PARAMETERS['ATTRIBUTION_WINDOW_DAYS']
POST_ATTENTION_HALF_LIFE_DAYS = PARAMETERS['POST_ATTENTION_HALF_LIFE_DAYS']  # только информация о сохранённых данных
TOP_K = 3
HORIZONS_DAYS = [7, 14]  # при необходимости добавьте 3
REPORT_HORIZON_DAYS = 7  # окно итоговой таблицы и выбора лидеров
assert REPORT_HORIZON_DAYS in HORIZONS_DAYS
assert HORIZONS_DAYS == sorted(set(HORIZONS_DAYS)) and min(HORIZONS_DAYS) > 0
OUTPUT = ROOT/'outputs/experiments/event_attribution'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})
def save(fig, name):
    fig.savefig(OUTPUT/(name+'.png'), dpi=145, bbox_inches='tight')
    fig.savefig(OUTPUT/(name+'.svg'), bbox_inches='tight')
    display(Image(filename=str(OUTPUT/(name+'.png'))))
    plt.close(fig)


## 1. Ручные ответы и граничные случаи

Каждый пример выполняется для четырёх моделей. Ожидаемая выручка задана явно, независимо от формул проверяемой функции. Для денежных сумм сравнение допускает только численную погрешность float (0,000001 ₽), а не округление до рубля. Проверка двух курсов начинается с двух позиций разной стоимости и одного заказа.


In [ ]:
T = pd.Timestamp('2026-09-01 12:00', tz='Europe/Moscow')
def order(oid='o', time=T, amount=100., user='u'):
    return dict(order_id=oid,user_id_hash=user,timestamp=time,amount=amount,variable_costs=0.)
def touch(i, days=1, placement=None, user='u', time=None):
    return dict(touch_id=i,user_id_hash=user,placement_id=placement or f'p{i}',timestamp=time if time is not None else T-pd.Timedelta(days=days))
def same(expected):return {m:expected for m in MODELS}
cases = [
 ('Одно касание',[touch(1)],[order()],same({'o':{'p1':100}})),
 ('Два касания',[touch(1,2),touch(2,1)],[order()],{'first':{'o':{'p1':100}},'last':{'o':{'p2':100}},'linear':{'o':{'p1':50,'p2':50}},'position':{'o':{'p1':50,'p2':50}}}),
 ('Три касания',[touch(1,3),touch(2,2),touch(3,1)],[order(amount=300)],{'first':{'o':{'p1':300}},'last':{'o':{'p3':300}},'linear':{'o':{'p1':100,'p2':100,'p3':100}},'position':{'o':{'p1':120,'p2':60,'p3':120}}}),
 ('Четыре касания',[touch(i,5-i) for i in range(1,5)],[order()],{'first':{'o':{'p1':100}},'last':{'o':{'p4':100}},'linear':{'o':{f'p{i}':25 for i in range(1,5)}},'position':{'o':{'p1':40,'p2':10,'p3':10,'p4':40}}}),
 ('Нет касаний',[],[order()],same({'o':{'NO_TOUCH':100}})),
 ('Старше окна',[touch(1,time=T-pd.Timedelta(days=30,seconds=1))],[order()],same({'o':{'NO_TOUCH':100}})),
 ('Ровно 30 дней',[touch(1,30)],[order()],same({'o':{'p1':100}})),
 ('После оплаты',[touch(1,time=T+pd.Timedelta(seconds=1))],[order()],same({'o':{'NO_TOUCH':100}})),
 ('Ровно в оплату',[touch(1,time=T)],[order()],same({'o':{'p1':100}})),
 ('Чужой неоплаченный путь',[touch(1),touch(2,user='nonbuyer')],[order()],same({'o':{'p1':100}})),
 ('Без рекламной метки',[dict(touch(1),placement_id=None)],[order()],same({'o':{'NO_TOUCH':100}})),
 ('Повторная оплата, сброс',[touch(1,3),touch(2,2),touch(3,1)],[order('old',T-pd.Timedelta(days=2),200),order()],{'first':{'old':{'p1':200},'o':{'p3':100}},'last':{'old':{'p2':200},'o':{'p3':100}},'linear':{'old':{'p1':100,'p2':100},'o':{'p3':100}},'position':{'old':{'p1':100,'p2':100},'o':{'p3':100}}}),
 ('Повтор без новых касаний',[touch(1,3)],[order('old',T-pd.Timedelta(days=2),200),order()],same({'old':{'p1':200},'o':{'NO_TOUCH':100}})),
 ('Одно время, порядок ID',[touch(2,1),touch(1,1)],[order()],{'first':{'o':{'p1':100}},'last':{'o':{'p2':100}},'linear':{'o':{'p1':50,'p2':50}},'position':{'o':{'p1':50,'p2':50}}}),
 ('Повторные касания поста',[touch(1,3,placement='p1'),touch(2,2,placement='p1'),touch(3,1)],[order(amount=300)],{'first':{'o':{'p1':300}},'last':{'o':{'p3':300}},'linear':{'o':{'p1':200,'p3':100}},'position':{'o':{'p1':180,'p3':120}}}),
 ('Два курса, один заказ',[touch(1)],[order(amount=sum([70.,30.]))],same({'o':{'p1':100}})),
]
checks=[]
for name, ts, os, expected in cases:
    tt=pd.DataFrame(ts,columns=['touch_id','user_id_hash','placement_id','timestamp'])
    tt['timestamp']=pd.to_datetime(tt.timestamp,utc=True).dt.tz_convert('Europe/Moscow')
    oo=pd.DataFrame(os)
    for model in MODELS:
        a=compute(tt,oo,model,30)  # ручные примеры заданы для окна 30 дней
        actual=a.assign(placement_id=a.placement_id.fillna('NO_TOUCH')).groupby(['order_id','placement_id']).attributed_revenue.sum().to_dict()
        target={(oid,p):v for oid,ps in expected[model].items() for p,v in ps.items()}
        assert actual.keys()==target.keys(), (name,model,actual,target)
        for key in target:assert np.isclose(actual[key],target[key],rtol=0,atol=1e-6),(name,model,key)
        assert np.allclose(a.groupby('order_id').weight.sum(),1,rtol=0,atol=1e-10)
        checks.append(dict(case=name,model=model,result='PASS'))
checks=pd.DataFrame(checks)
display(checks.pivot(index='case',columns='model',values='result'))
checks.to_csv(OUTPUT/'manual_checks.csv',index=False)
print(f'{len(checks)} проверок пройдено. Разные заказы с одинаковым временем не входят: правило не определено.')


## 2. Проверка сохранённой выборки

Читаем orders.csv, touches.csv, placements.csv и manifest.json из DATA_DIR. Параметры и границы наблюдения берутся из manifest.json; код генератора в основном запуске не выполняется. Входные файлы не перезаписываются. Результаты сохраняются в `outputs/demo/attribution/`.

Для каждого заказа проверяем сохранение выручки, сумму весов и допустимость касаний. Рейтинги по возрастным окнам считаются после полной атрибуции без перераспределения долей. Все модели работают с одними сохранёнными данными.


In [ ]:
def load_saved_data(directory, manifest):
    orders = pd.read_csv(directory/'orders.csv')
    touches = pd.read_csv(directory/'touches.csv')
    posts = pd.read_csv(directory/'placements.csv')
    timezone = manifest['parameters']['TIMEZONE']
    for frame, columns in [(orders,['timestamp','previous_order_time']), (touches,['timestamp']), (posts,['publication_time'])]:
        for column in columns:
            frame[column] = pd.to_datetime(frame[column],utc=True,format='mixed').dt.tz_convert(timezone)
    assert orders.order_id.notna().all() and orders.order_id.is_unique
    assert touches.touch_id.notna().all() and touches.touch_id.is_unique
    assert posts.placement_id.notna().all() and posts.placement_id.is_unique
    assert touches.placement_id.isin(posts.placement_id).all()
    assert orders[['user_id_hash','timestamp','amount','amount_cents']].notna().all().all()
    assert touches[['user_id_hash','timestamp']].notna().all().all()
    assert posts.publication_time.notna().all()
    assert len(orders) == manifest['historical_order_count']
    assert int(orders.amount_cents.sum()) == manifest['historical_revenue_cents']
    assert np.allclose(orders.amount*100,orders.amount_cents,rtol=0,atol=1e-6)
    ordered=orders.sort_values(['user_id_hash','timestamp'])
    expected_previous=ordered.groupby('user_id_hash').timestamp.shift()
    assert (ordered.previous_order_time.eq(expected_previous) | (ordered.previous_order_time.isna() & expected_previous.isna())).all()
    start=pd.Timestamp(manifest['observation_start']).tz_convert(timezone)
    end=pd.Timestamp(manifest['observation_end_exclusive']).tz_convert(timezone)
    for frame in [orders,touches]:assert (frame.timestamp.ge(start)&frame.timestamp.lt(end)).all()
    assert touches.timestamp.ge(touches.placement_id.map(posts.set_index('placement_id').publication_time)).all()
    return dict(orders=orders,touches=touches,posts=posts,observation_start=start,observation_end=end)

base=load_saved_data(DATA_DIR,input_manifest)
input_hashes={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['orders.csv','touches.csv','placements.csv','manifest.json']}
print('Загружены сохранённые данные:',len(base['orders']),'заказов,',len(base['touches']),'касаний.')
print('Параметры генерации из manifest.json:', 'seed =',BASE_SEED,'полураспад =',POST_ATTENTION_HALF_LIFE_DAYS)
placements=base['posts'].set_index('placement_id')
ids=placements.index.sort_values()
def calculate(ns):
    oo=ns['orders'].assign(variable_costs=np.nan)
    tt=ns['touches'][['touch_id','user_id_hash','placement_id','timestamp']]
    assert oo.order_id.is_unique and tt.touch_id.is_unique
    assert not oo.duplicated(['user_id_hash','timestamp']).any()
    assert oo.amount_cents.sum()==base['orders'].amount_cents.sum()
    assert tt.placement_id.isin(ids).all()
    order_map=oo.set_index('order_id');touch_map=tt.set_index('touch_id')
    results={}
    for model in MODELS:
        a=compute(tt,oo,model,WINDOW_DAYS)
        assert np.allclose(a.groupby('order_id').weight.sum().reindex(order_map.index),1,rtol=0,atol=1e-10)
        assert np.allclose(a.groupby('order_id').attributed_revenue.sum().reindex(order_map.index),order_map.amount,rtol=0,atol=1e-6)
        used=a[a.touch_id.notna()]
        ot=used.order_id.map(order_map.timestamp); t=used.touch_id.map(touch_map.timestamp)
        prev=used.order_id.map(order_map.previous_order_time)
        assert ((t<=ot)&(t>=ot-pd.Timedelta(days=WINDOW_DAYS))&(prev.isna()|t.gt(prev))).all()
        assert used.order_id.map(order_map.user_id_hash).equals(used.touch_id.map(touch_map.user_id_hash))
        results[model]=a
    return results
baseline=calculate(base)
for model,a in baseline.items():
    buyers=base['touches'][base['touches'].user_group.eq('buyer')]
    alt=compute(buyers,base['orders'].assign(variable_costs=np.nan,lead_id=None),model,WINDOW_DAYS)
    pd.testing.assert_frame_equal(a.reset_index(drop=True),alt.reset_index(drop=True))
def revenues(result):
    return pd.DataFrame({m:a.groupby('placement_id').attributed_revenue.sum().reindex(ids,fill_value=0) for m,a in result.items()})
base_revenue=revenues(baseline)
base_revenue.to_csv(OUTPUT/'baseline_revenue.csv')

no_touch=pd.Series({m:a.loc[a.touch_id.isna(),'attributed_revenue'].sum() for m,a in baseline.items()},name='Без сгенерированных касаний, ₽')

# Срез по времени заказа после полной атрибуции, без перераспределения долей.
def horizon_revenues(ns, result, horizon):
    published = placements.publication_time
    ends = published + pd.Timedelta(days=horizon)
    complete = published.ge(ns['observation_start']) & ends.le(ns['observation_end'])
    order_time = ns['orders'].set_index('order_id').timestamp
    columns = {}
    for model, a in result.items():
        start = a.placement_id.map(published)
        end = a.placement_id.map(ends)
        time = a.order_id.map(order_time)
        keep = time.ge(start) & time.lt(end)
        value = a.loc[keep].groupby('placement_id').attributed_revenue.sum().reindex(ids, fill_value=0.)
        value.loc[~complete.reindex(ids)] = np.nan
        columns[model] = value
    return pd.DataFrame(columns)

# Все окна сравниваются на общей когорте полного максимального горизонта.
common_ids = ids[(placements.publication_time.reindex(ids).ge(base['observation_start']) &
                 (placements.publication_time.reindex(ids)+pd.Timedelta(days=max(HORIZONS_DAYS))).le(base['observation_end'])).to_numpy()]
assert len(common_ids) >= TOP_K
coverage=[]
for h in HORIZONS_DAYS:
    v=horizon_revenues(base,baseline,h)
    coverage.append(dict(horizon_days=h,total_posts=len(ids),complete_posts=int(v[MODELS[0]].notna().sum()),common_posts=len(common_ids)))
pd.DataFrame(coverage).to_csv(OUTPUT/'horizon_coverage.csv',index=False)

# Проверка границ и отсутствия перенормировки на независимом примере.
probe_id=common_ids[0]; t0=placements.loc[probe_id,'publication_time']
probe_orders=pd.DataFrame({'order_id':[1,2,3,4], 'timestamp':[t0,t0+pd.Timedelta(days=3)-pd.Timedelta(seconds=1),t0+pd.Timedelta(days=3),t0-pd.Timedelta(seconds=1)]})
probe_a=pd.DataFrame({'order_id':[1,2,3,4], 'placement_id':[probe_id]*4,'attributed_revenue':[25.,50.,100.,200.]})
probe_ns=dict(base,orders=probe_orders)
assert horizon_revenues(probe_ns,{m:probe_a for m in MODELS},3).loc[probe_id].eq(75.).all()
probe_ns=dict(probe_ns,observation_end=t0+pd.Timedelta(days=3)-pd.Timedelta(seconds=1))
assert horizon_revenues(probe_ns,{m:probe_a for m in MODELS},3).loc[probe_id].isna().all()

def horizon_rows(ns,result,config,seed):
    rows=[]
    previous=None
    for h in HORIZONS_DAYS:
        values=horizon_revenues(ns,result,h).loc[common_ids]
        assert values.notna().all().all()
        if previous is not None: assert (values.to_numpy()+1e-7 >= previous.to_numpy()).all()
        previous=values
        for model in MODELS:
            ranks=values[model].rank(ascending=False,method='first')
            for pid in common_ids:
                rows.append(dict(config=config,seed=seed,horizon_days=h,model=model,placement_id=pid,revenue=values.loc[pid,model],rank=ranks[pid],top3=bool(ranks[pid]<=TOP_K)))
    return rows
baseline_horizons=pd.DataFrame(horizon_rows(base,baseline,'Сохранённая выборка',BASE_SEED))
baseline_horizons.to_csv(OUTPUT/'horizon_baseline.csv',index=False)


## 3. Событийная проверка

Используем исходные веса полной атрибуции. Выручку нужных продуктов выделяем по позициям заказа. Доля поста считается от всей выручки выбранных курсов периода, включая покупки без касаний. Рейтинг включает все посты, опубликованные до конца периода; одинаковая выручка даёт одинаковое место. Нулевой вклад сам по себе не означает поддержку гипотезы, даже при равенстве рангов.

Т-Банк: включены Аналитика ПРО, ML ПРО, Backend ПРО и Алгоритмы ПРО; AI-агенты исключены. Периоды задаются в МСК с включённой левой и исключённой правой границей. Сдвиг ±1 день — проверка чувствительности, без выбора «наилучшего» окна.


In [ ]:
# Событийная проверка: используются веса baseline, рассчитанные на полной истории.
EVENT_OUTPUT = OUTPUT
EVENT_OUTPUT.mkdir(parents=True, exist_ok=True)
TIMEZONE = PARAMETERS['TIMEZONE']
EVENTS = [
 dict(key='start_discount', name='Завершение скидок СТАРТ', start='2026-08-09', end='2026-08-12', event_date='2026-08-10', posts=[1836,1837,1838,1840], products=['Аналитика старт','ML старт','Backend старт','Алгоритмы старт']),
 dict(key='pro_launch', name='Анонс линейки ПРО', start='2026-08-22', end='2026-08-29', event_date='2026-08-22', posts=[1859], products=['Аналитика про','ML про','Backend про','Алгоритмы про','AI агенты']),
 dict(key='tbank_deadline', name='Дедлайн отбора в Т-Банк', start='2026-08-31', end='2026-09-07', event_date='2026-09-06', posts=[1878,1880], products=['Аналитика про','ML про','Backend про','Алгоритмы про']),
]
# Т-Банк: четыре направления подготовки; AI-агенты в этот срез не включены.
# Дополнительные ранние публикации остаются конкурентами в общем рейтинге.
items = pd.read_csv(DATA_DIR/'order_items.csv')
input_hashes['order_items.csv'] = hashlib.sha256((DATA_DIR/'order_items.csv').read_bytes()).hexdigest()
assert items.item_id.is_unique and items.order_id.isin(base['orders'].order_id).all()
item_totals=items.groupby('order_id').amount_cents.sum().reindex(base['orders'].order_id)
assert np.array_equal(item_totals.to_numpy(),base['orders'].amount_cents.to_numpy())
assert np.allclose(items.amount*100,items.amount_cents,rtol=0,atol=1e-6)
item_data=items.merge(base['orders'][['order_id','timestamp']],on='order_id',validate='many_to_one')
post_ids=placements.post_id.astype(int)

def event_slice(event, shift=0):
    start=pd.Timestamp(event['start'],tz=TIMEZONE)+pd.Timedelta(days=shift)
    end=pd.Timestamp(event['end'],tz=TIMEZONE)+pd.Timedelta(days=shift)
    selected=item_data[item_data.course.isin(event['products']) & item_data.timestamp.ge(start) & item_data.timestamp.lt(end)]
    amounts=selected.groupby('order_id').amount_cents.sum()/100
    denominator=amounts.sum()
    group_ids=placements.index[post_ids.isin(event['posts'])]
    assert len(group_ids)==len(event['posts'])
    # Все публикации, которые по времени могли предшествовать покупке в периоде.
    candidates=placements.index[placements.publication_time.lt(end)]
    summaries=[];post_rows=[]
    for model,a in baseline.items():
        credit=a[a.order_id.isin(amounts.index)].copy()
        credit['event_revenue']=credit.weight*credit.order_id.map(amounts)
        assert np.isclose(credit.event_revenue.sum(),denominator,atol=1e-6,rtol=0)
        used=credit[credit.placement_id.notna()]
        order_time=credit.order_id.map(base['orders'].set_index('order_id').timestamp)
        pub=credit.placement_id.map(placements.publication_time)
        assert order_time[pub.notna()].ge(pub[pub.notna()]).all()
        values=used.groupby('placement_id').event_revenue.sum().reindex(candidates,fill_value=0.)
        ranks=values.rank(ascending=False,method='min') # равная выручка — одинаковое место
        group_revenue=values.reindex(group_ids,fill_value=0.).sum()
        no_touch=credit.loc[credit.placement_id.isna(),'event_revenue'].sum()
        assert np.isclose(values.sum()+no_touch,denominator,rtol=0,atol=1e-6)
        positive=values[values>0]
        leaders=positive[positive.eq(positive.max())].index if len(positive) else []
        summaries.append(dict(event=event['key'],name=event['name'],shift_days=shift,model=model,start=start,end_exclusive=end,
            period_complete=bool(start>=base['observation_start'] and end<=base['observation_end']),
            total_revenue=denominator,orders=len(amounts),group_revenue=group_revenue,
            group_share_pct=100*group_revenue/denominator if denominator else np.nan,
            no_touch_revenue=no_touch,leader_posts=', '.join(str(int(placements.loc[p,'post_id'])) for p in leaders)))
        for pid in candidates:
            post_rows.append(dict(event=event['key'],shift_days=shift,model=model,placement_id=pid,post_id=int(placements.loc[pid,'post_id']),
                selected=pid in group_ids,revenue=values[pid],rank=ranks[pid],
                share_pct=100*values[pid]/denominator if denominator else np.nan))
    return summaries,post_rows

summary_rows=[];post_rows=[]
for event in EVENTS:
    assert set(event['products']).issubset(set(items.course))
    for shift in [-1,0,1]:
        summary_part,post_part=event_slice(event,shift)
        summary_rows.extend(summary_part);post_rows.extend(post_part)
event_runs=pd.DataFrame(summary_rows)
event_post_runs=pd.DataFrame(post_rows)
event_summary=event_runs[event_runs.shift_days.eq(0)].copy()
event_posts=event_post_runs[event_post_runs.shift_days.eq(0)].copy()
event_summary.to_csv(EVENT_OUTPUT/'event_summary.csv',index=False)
event_posts.to_csv(EVENT_OUTPUT/'event_posts.csv',index=False)
event_runs.to_csv(EVENT_OUTPUT/'window_sensitivity.csv',index=False)
event_post_runs.to_csv(EVENT_OUTPUT/'post_window_sensitivity.csv',index=False)

# Одна строка на связанный пост; доля и ранг описывают один и тот же срез продуктов.
selected_posts=event_posts[event_posts.selected]
event_comparison=selected_posts.pivot(index=['event','post_id'],columns='model',values=['revenue','share_pct','rank'])
event_comparison.to_csv(EVENT_OUTPUT/'selected_post_comparison.csv')
readable=selected_posts.assign(result=lambda x:x['rank'].astype(int).astype(str)+' место · '+x.share_pct.map(lambda v:f'{v:.1f}%'))
readable.loc[readable.revenue.eq(0),'result']='0 ₽ · нет вклада'
print('Выбранные посты: место среди всех опубликованных постов и доля выручки нужных курсов.')
display(readable.pivot(index=['event','post_id'],columns='model',values='result').reindex(columns=MODELS))
print('Вклад выбранной группы постов; знаменатель включает покупки без касаний.')
display(event_summary[['name','model','total_revenue','group_revenue','group_share_pct','leader_posts']].round(2))
print('Чувствительность доли группы к сдвигу всего периода на −1 / 0 / +1 день:')
display(event_runs.pivot(index=['name','model'],columns='shift_days',values='group_share_pct').round(2))

# Линейная модель сохраняет положительный вес для каждого допустимого касания.
# Поэтому её строки позволяют проверить наличие путей, независимо от выбора победителя.
path_rows=[]
for event in EVENTS:
    start=pd.Timestamp(event['start'],tz=TIMEZONE);end=pd.Timestamp(event['end'],tz=TIMEZONE)
    selection=item_data[item_data.course.isin(event['products']) & item_data.timestamp.ge(start) & item_data.timestamp.lt(end)]
    a=baseline['linear'][baseline['linear'].order_id.isin(selection.order_id)]
    for post in event['posts']:
        pid=placements.index[post_ids.eq(post)][0]
        eligible=a[a.placement_id.eq(pid)]
        path_rows.append(dict(event=event['key'],post_id=post,orders_with_eligible_touch=eligible.order_id.nunique(),eligible_touches=len(eligible)))
path_diagnostics=pd.DataFrame(path_rows)
path_diagnostics.to_csv(EVENT_OUTPUT/'selected_post_paths.csv',index=False)
print('Есть ли в сохранённых путях покупателей допустимые касания выбранных постов:')
display(path_diagnostics)


In [ ]:
import matplotlib.dates as mdates
colors=['#3478AB','#D58536','#31847D','#8872AC']
model_names=['Первое касание','Последнее касание','Линейная','Позиционная']
daily_rows=[];coverage_rows=[]
for event in EVENTS:
    start=pd.Timestamp(event['start'],tz=TIMEZONE);end=pd.Timestamp(event['end'],tz=TIMEZONE)
    context_start=start-pd.Timedelta(days=7)
    visible_start=max(context_start,base['observation_start']);visible_end=min(end,base['observation_end'])
    days=pd.date_range(visible_start,visible_end-pd.Timedelta(days=1),freq='D')
    product_items=item_data[item_data.course.isin(event['products'])]
    daily=(product_items.groupby(product_items.timestamp.dt.normalize()).amount_cents.sum()/100).reindex(days,fill_value=0.)
    for day,value in daily.items(): daily_rows.append(dict(event=event['key'],date=day,revenue=value,in_event=bool(start<=day<end)))
    prior=daily[daily.index<start];period=daily[(daily.index>=start)&(daily.index<end)]
    coverage_rows.append(dict(event=event['key'],prior_days=len(prior),prior_requested_days=7,period_days=len(period),
        prior_mean_daily=prior.mean(),period_mean_daily=period.mean(),peak_date=period.idxmax(),peak_revenue=period.max()))
    fig,axs=plt.subplots(3,1,figsize=(11,8),gridspec_kw={'height_ratios':[3,1.3,2.4]},layout='constrained')
    fig.suptitle(event['name'],fontsize=19,weight='bold')
    ax,timeline,bars=axs
    ax.bar(daily.index,daily.values/1000,width=.8,color=['#AEBBC8' if d<start else '#3478AB' for d in daily.index])
    ax.axvspan(start,end,color='#3478AB',alpha=.06)
    ax.axvline(pd.Timestamp(event['event_date'],tz=TIMEZONE),color='#AA602C',ls='--',lw=1.4,label='Дата события / дедлайн')
    ax.set(ylabel='Выручка курсов, тыс. ₽',title=f'Серым — предшествующие дни ({len(prior)} из 7 доступны); синим — период оценки')
    ax.legend(loc='upper left',fontsize=9);ax.grid(axis='x',visible=False)
    selected=placements[post_ids.isin(event['posts'])].sort_values('publication_time')
    for j,row in enumerate(selected.itertuples()):
        timeline.scatter(row.publication_time,j,color='#AA602C',s=45)
        right_side=row.publication_time > visible_start+(visible_end-visible_start)/2
        label_x=row.publication_time+pd.Timedelta(hours=-3 if right_side else 3)
        timeline.text(label_x,j,f'№{int(row.post_id)} · {row.publication_time:%d.%m %H:%M}',fontsize=9,va='center',ha='right' if right_side else 'left')
    timeline.set(yticks=[],ylabel='Посты',ylim=(-.8,len(selected)-.2))
    for a in [ax,timeline]:
        a.set_xlim(visible_start-pd.Timedelta(hours=12),visible_end)
        a.xaxis.set_major_locator(mdates.DayLocator(interval=2,tz=TIMEZONE))
        a.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m',tz=TIMEZONE))
    timeline.grid(False);timeline.set_xlabel('Дата, 2026 год (МСК)')
    r=event_summary[event_summary.event.eq(event['key'])].set_index('model').reindex(MODELS)
    bars.barh(model_names,r.group_share_pct,color=colors,height=.58)
    for i,v in enumerate(r.group_share_pct): bars.text(v+1,i,f'{v:.1f}%'.replace('.',','),va='center',fontsize=11,weight='bold')
    bars.invert_yaxis();bars.set_xlim(0,105);bars.set_xticks([0,25,50,75,100])
    bars.set(xlabel='Доля выручки периода, приписанная выбранной группе постов, %')
    bars.set_title('Сравнение моделей на одних сохранённых касаниях',loc='left',fontsize=12)
    bars.grid(axis='y',visible=False)
    fig.supxlabel('Продажи исторические, касания синтетические. Связь по времени не устанавливает причинный эффект.',fontsize=9)
    fig.savefig(EVENT_OUTPUT/(event['key']+'.png'),dpi=150,bbox_inches='tight')
    display(Image(filename=str(EVENT_OUTPUT/(event['key']+'.png'))));plt.close(fig)
event_daily=pd.DataFrame(daily_rows);event_context=pd.DataFrame(coverage_rows)
event_daily.to_csv(EVENT_OUTPUT/'event_daily.csv',index=False)
event_context.to_csv(EVENT_OUTPUT/'event_context.csv',index=False)
print('Дневная выручка для контекста; разность средних не является оценкой эффекта:')
display(event_context.round(2))
# Один порядок весов для всех позиций заказа: отбор продуктов не меняет касания.
# Отчёт отдельно от D7/D14, без ограничения общей когорты зрелости.
for name,digest in input_hashes.items():
    assert hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()==digest
(EVENT_OUTPUT/'manifest.json').write_text(json.dumps(dict(events=EVENTS,input_sha256=input_hashes,
    source='saved_csv',window_days=WINDOW_DAYS,seed=BASE_SEED,shift_days=[-1,0,1],
    tie_rule='min: равная выручка — одинаковый ранг',rank_population='все посты с публикацией до конца периода',
    product_allocation='сумма выбранных позиций заказа × исходные веса полной атрибуции',
    share_denominator='вся выручка выбранных позиций периода, включая покупки без касаний'),ensure_ascii=False,indent=2))
print('Событийные результаты:',EVENT_OUTPUT)
print('Контрольные суммы входных файлов не изменились.')


In [ ]:
from IPython.display import Markdown
lines=['### Выводы событийной проверки','']
for event in EVENTS:
    s=event_summary[event_summary.event.eq(event['key'])]
    c=event_context.set_index('event').loc[event['key']]
    paths=path_diagnostics[path_diagnostics.event.eq(event['key'])]
    zero=paths.loc[paths.orders_with_eligible_touch.eq(0),'post_id'].astype(str).tolist()
    lines.append(f"- **{event['name']}**: группе постов приписано {s.group_share_pct.min():.2f}–{s.group_share_pct.max():.2f}% выручки периода. Пик выбранных продаж — {pd.Timestamp(c.peak_date):%d.%m}." + (f" Без допустимых касаний в выбранных покупках: №{', №'.join(zero)}." if zero else ''))
lines.extend(['','Для проверки гипотезы смотрим на долю и место постов, но не назначаем победителя только по наибольшему совпадению с ожиданиями. Если сохранённый путь покупки не содержит допустимого касания поста, ни одна из четырёх моделей не может приписать ему выручку. Это ограничение входных данных, а не свидетельство отсутствия реального эффекта поста.',''])
for event in EVENTS:
    peak=pd.Timestamp(event_context.set_index('event').loc[event['key'],'peak_date'])
    later=placements[post_ids.isin(event['posts']) & placements.publication_time.ge(peak+pd.Timedelta(days=1))]
    if len(later):lines.append(f"- {event['name']}: посты №"+', №'.join(later.post_id.astype(int).astype(str))+f" опубликованы после дня максимальных продаж ({peak:%d.%m}) и не могут объяснять предшествующие покупки.")
lines.extend(['','**Устойчивость:** сдвиг календарного окна на один день показан в таблице выше; результаты не усредняются и наиболее удобное окно не выбирается. Для скидок доступно только пять из семи предшествующих дней. Различия дневной выручки описательные, без оценки причинного эффекта.','', '**Что сохраняем:** ручные проверки контролируют корректность расчётов; D7/D14 сравнивают одинаковый возраст постов. Эти разделы остаются дополнительной диагностикой. Событийный анализ отвечает на новую гипотезу, но синтетические касания не позволяют выбрать лучшую модель реального влияния. Новая генерация для этого запуска не выполнялась.'])
event_conclusions='\n'.join(lines)
display(Markdown(event_conclusions))
(EVENT_OUTPUT/'conclusions.md').write_text(event_conclusions+'\n')


## Дополнительная диагностика: D7/D14

Сравнение одинакового возраста постов и календарь лидеров сохранены для диагностики. Они не служат ответом на событийную гипотезу.


In [ ]:
# Одна таблица: реальные категории и даты, модельная выручка за выбранный возраст.
values=horizon_revenues(base,baseline,REPORT_HORIZON_DAYS).loc[common_ids]
report=placements.loc[common_ids,['post_id','publication_time','post_type']].copy()
for model in MODELS:report[model]=values[model]
# Для вывода — по средней приписанной выручке четырёх моделей; это только порядок строк.
report=report.loc[values.mean(axis=1).sort_values(ascending=False,kind='stable').index]
report.to_csv(OUTPUT/'placement_comparison.csv',index_label='placement_id')
shown=report.copy()
shown['publication_time']=shown.publication_time.dt.strftime('%d.%m.%Y %H:%M')
shown=shown.rename(columns={'post_id':'Пост','publication_time':'Публикация (МСК)','post_type':'Категория'})
shown[MODELS]=shown[MODELS].round(2)
print(f'Приписанная выручка за первые {REPORT_HORIZON_DAYS} дней, ₽. Полная таблица: {len(shown)} постов.')
with pd.option_context('display.max_rows',None,'display.max_columns',None):display(shown.reset_index(drop=True))
print('Покрытие окон:')
display(pd.DataFrame(coverage))

# Лидеры — объединение топ-3 каждой модели в выбранном окне, с теми же правилами равенства.
ranked=baseline_horizons[baseline_horizons.horizon_days.eq(REPORT_HORIZON_DAYS)]
leader_ids=ranked.loc[ranked['rank'].le(TOP_K),'placement_id'].unique()
leader_posts=placements.loc[leader_ids].sort_values('publication_time')
daily=base['orders'].groupby(base['orders'].timestamp.dt.normalize()).amount.sum()
days=pd.date_range(base['observation_start'],base['observation_end']-pd.Timedelta(days=1),freq='D')
daily=daily.reindex(days,fill_value=0.)
assert np.isclose(daily.sum(),base['orders'].amount.sum(),rtol=0,atol=1e-6)
daily.rename('recorded_revenue').rename_axis('date').to_csv(OUTPUT/'calendar_sales.csv')
leader_posts[['post_id','publication_time','post_type']].to_csv(OUTPUT/'calendar_leaders.csv')
import matplotlib.dates as mdates
fig,ax=plt.subplots(figsize=(11,4.8),layout='constrained')
ax.bar(daily.index,daily.to_numpy()/1000,width=.8,color='#4C72B0',label='Исторические покупки')
for row in leader_posts.itertuples():
    ax.axvline(row.publication_time,color='#B15C21',linestyle='--',linewidth=1,alpha=.8)
    ax.text(row.publication_time,.97,f' {int(row.post_id)}',rotation=90,ha='right',va='top',transform=ax.get_xaxis_transform(),fontsize=10)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m',tz=base['observation_start'].tz))
ax.set(ylabel='Выручка по покупкам, тыс. ₽',xlabel='Дата (МСК)',ylim=(0,daily.max()/1000*1.25),
       title=f'Календарь продаж и публикации участников топ-3 на D{REPORT_HORIZON_DAYS}')
ax.plot([],[],color='#B15C21',linestyle='--',label='Публикации лидеров хотя бы одной модели')
ax.legend(loc='upper right',fontsize=9)
fig.supxlabel('Покупки из истории; лидеры определены по сгенерированным касаниям. Совпадение не доказывает влияние.',fontsize=9)
save(fig,'calendar_sales_and_leaders')
manifest={'experiment_only':True,'input_mode':'saved_csv','input_directory':str(DATA_DIR.relative_to(ROOT)),
          'input_sha256':input_hashes,'generation_parameters':PARAMETERS,
          'window_days':WINDOW_DAYS,'horizons_days':HORIZONS_DAYS,'report_horizon_days':REPORT_HORIZON_DAYS,
          'common_placement_ids':common_ids.tolist(),'manual_checks_passed':len(checks),
          'engine_sha256':hashlib.sha256((ROOT/'src/core/attribution.py').read_bytes()).hexdigest(),
          'as_of':pd.Timestamp.now(tz='UTC').isoformat()}
(OUTPUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
print('Общая выручка:',round(base['orders'].amount.sum(),2),'₽')
print('Без подходящих касаний:',round(no_touch.iloc[0],2),'₽')


## Как читать результат

Таблица сравнивает четыре правила распределения выручки. Категории публикаций — исходная разметка, которая может содержать ошибки. Они не меняют расчёт. Равенство результатов моделей не доказывает эффективность поста.

D7/D14 отсчитываются от публикации по времени покупки: [публикация, публикация + H). Веса рассчитываются до среза и не перенормируются. Во всех окнах сравнивается один набор постов с полностью наблюдаемым максимальным горизонтом; при его изменении состав может поменяться. Отсутствие событий в полном окне означает нулевую выручку.

Календарь помогает заметить совпадение публикаций с всплесками покупок. Он не отделяет влияние поста от внешнего спроса. Частота попадания в топ-3 не рассчитывается: здесь одна сохранённая выборка. Изменение угасания требует нового запуска генератора.
